In [7]:
import os

os.environ["OPENAI_API_KEY"] = "<your-openai-key>"

print("API key set!")


API key set!


In [9]:
import os

# LangSmith monitoring setup (fill in your own key)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = "<your-langsmith-key>"  
os.environ["LANGCHAIN_PROJECT"] = "lumina-assignment"

print("LangSmith env set. Remember to insert your real API key.")


LangSmith env set. Remember to insert your real API key.


In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are Lumina, a compliance intelligence assistant."),
    ("human", "User question: {question}\n\nAnswer clearly and concisely.")
])

chain = prompt | llm | StrOutputParser()

resp = chain.invoke({"question": "Explain prompt engineering in simple terms."})
resp


"Prompt engineering is the process of designing and refining the questions or instructions given to an AI model to get the best possible responses. It involves choosing the right words and structure to guide the AI in generating accurate and relevant answers. Essentially, it's about crafting prompts that help the AI understand what you want it to do."

In [14]:
from utils.embed_store import PersistentEmbedStore

embed_store = PersistentEmbedStore()

prompt_with_context = ChatPromptTemplate.from_messages([
    ("system", "You are Lumina, a compliance intelligence assistant."),
    (
        "human",
        "Use ONLY the context below to answer.\n\n"
        "If the answer isn't in the context, say 'Not found in document.'\n\n"
        "Context:\n{context}\n\nQuestion:\n{question}"
    ),
])

chain_with_context = prompt_with_context | llm | StrOutputParser()

question = "What are the breach notification requirements?"

context = embed_store.query(question)

print("CONTEXT PREVIEW:\n", context[:500], "\n")
answer = chain_with_context.invoke({"question": question, "context": context})
answer


✅ Loaded existing store with 13 embeddings.
CONTEXT PREVIEW:
 nd appropriate.
8. Business Associates
8.1 Business associate arrangements include required assurances on safeguards
and breach reporting.
8.2 Subcontractors are held to comparable obligations.
9. Breach Notification
9.1 Potential breaches are assessed for compromise probability using
standardized factors.
9.2 Notices to individuals and authorities follow simulated timelines.
10. Documentation and Retention
10.1 Policies and records are retained for the required period in a
retrievable format.
1 



'The breach notification requirements include the following:\n\n1. Potential breaches are assessed for compromise probability using standardized factors.\n2. Notices to individuals and authorities follow simulated timelines.\n3. Incidents are triaged and documented with facts, effects, and remedial actions.\n4. Notices to authorities and data subjects follow statutory timelines when required.'

In [17]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class AppState(TypedDict):
    question: str
    context: str
    answer: str
    quality: Literal["unknown", "good", "bad"]
    attempts: int  # how many times we've generated an answer

# 1) Node: retrieve context from your FAISS store
def retrieve_node(state: AppState) -> AppState:
    ctx = embed_store.query(state["question"])
    return {**state, "context": ctx}

# 2) Node: generate answer using LangChain chain_with_context
def generate_node(state: AppState) -> AppState:
    answer = chain_with_context.invoke(
        {"question": state["question"], "context": state["context"]}
    )
    attempts = state.get("attempts", 0) + 1
    return {**state, "answer": answer, "quality": "unknown", "attempts": attempts}

# 3) Node: evaluate answer quality
def evaluate_node(state: AppState) -> AppState:
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a strict evaluator. Reply ONLY 'good' or 'bad'."),
        ("human", "Question: {question}\nContext: {context}\nAnswer: {answer}\nIs this answer good or bad?")
    ])
    eval_chain = eval_prompt | llm | StrOutputParser()
    quality_raw = eval_chain.invoke(state)
    quality = "good" if "good" in quality_raw.lower() else "bad"
    return {**state, "quality": quality}

def route_after_eval(state: AppState) -> str:
    # Retry only if it's bad AND we haven't retried too many times
    if state["quality"] == "bad" and state.get("attempts", 0) < 2:
        return "retry"
    return "end"

builder = StateGraph(AppState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.add_node("evaluate", evaluate_node)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", "evaluate")

builder.add_conditional_edges(
    "evaluate",
    route_after_eval,
    {
        "retry": "generate",  # 🔁 feedback loop / cycle
        "end": END,
    }
)

graph = builder.compile()
print("LangGraph compiled with bounded retries.")


✅ LangGraph compiled with bounded retries.


In [18]:
initial_state: AppState = {
    "question": "What are the breach notification requirements?",
    "context": "",
    "answer": "",
    "quality": "unknown",
    "attempts": 0,
}

final_state = graph.invoke(initial_state)
final_state


{'question': 'What are the breach notification requirements?',
 'context': "nd appropriate.\n8. Business Associates\n8.1 Business associate arrangements include required assurances on safeguards\nand breach reporting.\n8.2 Subcontractors are held to comparable obligations.\n9. Breach Notification\n9.1 Potential breaches are assessed for compromise probability using\nstandardized factors.\n9.2 Notices to individuals and authorities follow simulated timelines.\n10. Documentation and Retention\n10.1 Policies and records are retained for the required period in a\nretrievable format.\n10.2 Version control captures material changes and approvals.\n11. Contact\n11.1 Questions regarding this sample may be directed to the simulated Privacy\nOfficer contact listed in metadata.\n\ntable.\n6.2 Access follows least privilege and is reviewed periodically.\n7. Breach Notification\n7.1 Incidents are triaged and documented with facts, effects, and remedial\nactions.\n7.2 Notices to authorities and data

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1) Base prompt (already have chain_with_context, but let's use a simple one here)
base_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are Lumina, a compliance assistant."),
    ("human", "Question: {question}")
])
base_chain = base_prompt | llm | StrOutputParser()

# 2) Structured answer
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a compliance expert. Answer in 3 sections: Summary, Details, and Actionable Next Steps."),
    ("human", "Question: {question}")
])
structured_chain = structured_prompt | llm | StrOutputParser()

# 3) Risk-focused answer
risk_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a risk-focused compliance assistant. Emphasize risks and mitigations."),
    ("human", "Question: {question}")
])
risk_chain = risk_prompt | llm | StrOutputParser()

# 4) Checklist-style answer
checklist_prompt = ChatPromptTemplate.from_messages([
    ("system", "You produce clear compliance checklists."),
    ("human", "Question: {question}")
])
checklist_chain = checklist_prompt | llm | StrOutputParser()

q = "What should a bank do to securely handle customer financial data?"

base = base_chain.invoke({"question": q})
s1 = structured_chain.invoke({"question": q})
s2 = risk_chain.invoke({"question": q})
s3 = checklist_chain.invoke({"question": q})

print("BASE:\n", base[:600], "\n")
print("STRUCTURED:\n", s1[:600], "\n")
print("RISK-FOCUSED:\n", s2[:600], "\n")
print("CHECKLIST:\n", s3[:600], "\n")


BASE:
 To securely handle customer financial data, a bank should implement a comprehensive data security strategy that includes the following key practices:

1. **Data Encryption**: Encrypt sensitive data both in transit and at rest to protect it from unauthorized access.

2. **Access Controls**: Implement strict access controls to ensure that only authorized personnel can access customer financial data. Use role-based access and the principle of least privilege.

3. **Multi-Factor Authentication (MFA)**: Require MFA for both customers and employees to add an additional layer of security when accessi 

STRUCTURED:
 ### Summary
To securely handle customer financial data, a bank must implement a comprehensive data protection strategy that includes robust security measures, compliance with regulations, employee training, and regular audits. This ensures the confidentiality, integrity, and availability of sensitive information while maintaining customer trust and regulatory compliance.

##